# SISC TCN Training
- Short-term pattern classifier
- Train period: 2015-01-01 ~ 2023-12-31
- Output: model.pt / scaler.pkl / metadata.json

In [ ]:
import subprocess
result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', '20260326-#343-AI-kaggle-pipeline', 'https://github.com/SISC-IT/sisc-web.git', '/kaggle/working/sisc-web'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/sisc-web')
print('Path added')

In [ ]:
!pip install -q scikit-learn pandas numpy torch


import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
gpu_name = result.stdout.strip()
print(f'GPU: {gpu_name}')


In [ ]:
import os
import importlib.util

train_script = '/kaggle/working/sisc-web/AI/modules/signal/models/TCN/train_kaggle.py'
assert os.path.exists(train_script), f'Not found: {train_script}'

spec = importlib.util.spec_from_file_location('train_kaggle', train_script)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
import argparse
args = argparse.Namespace(
    parquet_dir  = '/kaggle/input/sisc-ai-trading-dataset',
    start_date   = '2015-01-01',
    end_date     = '2023-12-31',
    seq_len      = 60,
    epochs       = 20,
    batch_size   = 64,
    learning_rate= 1e-3,
    kernel_size  = 3,
    dropout      = 0.2,
    channels     = [32, 64, 64],
    output_dir   = '/kaggle/working'
)
module.train_model(args)

In [ ]:
import os
for f in sorted(os.listdir('/kaggle/working')):
    p = f'/kaggle/working/{f}'
    if os.path.isfile(p):
        print(f'{f}: {os.path.getsize(p)/1024/1024:.2f} MB')

In [ ]:
import shutil
if os.path.exists('/kaggle/working/sisc-web'):
    shutil.rmtree('/kaggle/working/sisc-web')
    print("sisc-web 폴더 정리 완료")